

# What I tried to train GAN
- I used the Generator to generate the embeddings vectors.
- I calculated the AUC for each epoch to avoid learning instability, and used the model with the largest AUC for inference.
- ConvTranspose1d was used to save memory and increase dimensionality.

# Notes.
- The model was created by extracting layers from the bert-base-uncased of HuggingFace's BeertForSequenceClassification, but I have not confirmed whether this method is effective or not.
- The Real and Fake labels are reversed from those of a normal GAN: Real is trained as 0 and Fake as 1 according to the label "Generated".
- In a normal GAN, only Real data is included in the Real data set, but the training includes a very small number of Fake data.

# Future works
- Currently, GAN learning is not yet stable. There is a possibility to significantly improve the performance of the model by improving the network structure, loss, and training parameters.

In [ ]:
# Upload Kaggle API key = json file
from google.colab import files
files.upload()

# Move the file to the right folder and give permissions
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Download the dataset
!kaggle competitions download -c llm-detect-ai-generated-text


Saving kaggle.json to kaggle (1).json
403 - Forbidden - You must accept this competition's rules before you'll be able to download files.


In [ ]:
!unzip flower-color-images.zip

Traceback (most recent call last):
  File "/usr/local/bin/kaggle", line 4, in <module>
    from kaggle.cli import main
  File "/usr/local/lib/python3.11/dist-packages/kaggle/__init__.py", line 7, in <module>
    api.authenticate()
  File "/usr/local/lib/python3.11/dist-packages/kaggle/api/kaggle_api_extended.py", line 407, in authenticate
    raise IOError('Could not find {}. Make sure it\'s located in'
OSError: Could not find kaggle.json. Make sure it's located in /root/.config/kaggle. Or use the environment method. See setup instructions at https://github.com/Kaggle/kaggle-api/


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

In [ ]:
import random
import string

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

from transformers import BertTokenizer, BertForSequenceClassification
from transformers import BertConfig
from transformers.models.bert.modeling_bert import BertEncoder
from sklearn.metrics import roc_auc_score

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")


# Data loading

In [ ]:
# download the data you can find here :
TRAIN_PATH = 'train_essays.csv'
TEST_PATH = 'test_essays.csv'
PROMPT_PATH = 'train_prompts.csv'
#AIGEN_PATH = 'ai_generated_train_essays.csv'

src_train = pd.read_csv(TRAIN_PATH)
src_prompt = pd.read_csv(PROMPT_PATH)

src_sub = pd.read_csv(TEST_PATH)
#src_aigen = pd.read_csv(AIGEN_PATH)

# Model preparation

In [ ]:
tokenizer_save_path = 'bert-base-uncased'
model_save_path = 'bert-base-uncased'

tokenizer = BertTokenizer.from_pretrained(tokenizer_save_path)
pretrained_model = BertForSequenceClassification.from_pretrained(model_save_path)
embedding_model = pretrained_model.bert.embeddings

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


# Parameter definition

In [ ]:
train_batch_size = 2
test_batch_size = 2
lr = 0.0002
beta1 = 0.5
nz = 100  # Dimensions of the latent vector
num_epochs = 30
num_hidden_layers = 6
train_ratio = 0.9

# Data Preparation

In [ ]:
class GANDAIGDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]

all_num = src_train.shape[0]
train_num = int(all_num * train_ratio)
test_num = all_num - train_num


train_set = src_train.sample(train_num, random_state=123).reset_index(drop=True)
test_set = pd.concat([
    src_train.drop(train_set.index),
]).reset_index(drop=True)


train_dataset = GANDAIGDataset(train_set.text, train_set.generated)
test_dataset = GANDAIGDataset(test_set.text, test_set.generated)

train_loader = DataLoader(train_dataset, batch_size=train_batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=test_batch_size, shuffle=True)

# Generator definition

In [ ]:
config = BertConfig(num_hidden_layers=num_hidden_layers)

class Generator(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.fc = nn.Linear(input_dim, 256 * 128)

        self.conv_net = nn.Sequential(
            nn.ConvTranspose1d(256, 256, 4, stride=2),
            nn.BatchNorm1d(256),
            nn.ReLU(True),
            nn.ConvTranspose1d(256, 512, 4, stride=2),
            nn.ReLU(True),
            nn.ConvTranspose1d(512, 768, 3, stride=1)
        )
        self.bert_encoder = BertEncoder(config)


    def forward(self, x):
        x = self.fc(x)
        x = x.view(-1, 256, 128)
        x = self.conv_net(x)
        x = x.view(-1, 520, 768)
        x = x[:, :512, :768]
        x = self.bert_encoder(x)
        return x


# Discriminator definition

In [ ]:
class SumBertPooler(torch.nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, hidden_states: torch.Tensor) -> torch.Tensor:
        sum_hidden = hidden_states.sum(dim=1)
        sum_mask = sum_hidden.sum(1).unsqueeze(1)
        sum_mask = torch.clamp(sum_mask, min=1e-9)

        mean_embeddings = sum_hidden / sum_mask
        return mean_embeddings


class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert_encoder = BertEncoder(config)
        self.bert_encoder.layer = nn.ModuleList([
            layer for layer in pretrained_model.bert.encoder.layer[:6]
        ])
        self.pooler = SumBertPooler()
        self.classifier = torch.nn.Sequential(
            torch.nn.Linear(768, 64, bias=True),
            torch.nn.BatchNorm1d(64),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.3),
            torch.nn.Linear(64, 16, bias=True),
            torch.nn.BatchNorm1d(16),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.3),
            torch.nn.Linear(16, 1, bias=True)

        )

    def forward(self, input):
        out = self.bert_encoder(input)
        out = self.pooler(out.last_hidden_state)
        out = self.classifier(out)
        return torch.sigmoid(out).view(-1)

# Training

In [ ]:
def eval_auc(model):
    model.eval()

    predictions = []
    actuals = []
    with torch.no_grad():
        for batch in test_loader:
            encodings = tokenizer(batch[0], padding=True, truncation=True, return_tensors="pt")
            input_ids = encodings['input_ids']
            token_type_ids = encodings['token_type_ids']
            embeded = embedding_model(input_ids=input_ids, token_type_ids=token_type_ids)
            embeded = embeded.to(device)
            attention_mask = encodings['attention_mask'].to(device)
            label = batch[1].float().to(device)

            outputs = model(embeded)
            predictions.extend(outputs.cpu().numpy())
            actuals.extend(label.cpu().numpy())

    auc = roc_auc_score(actuals, predictions)
    print("AUC:", auc)
    return auc

def get_model_info_dict(model, epoch, auc_score):
    current_device = next(model.parameters()).device
    model.to('cpu')

    model_info = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'auc_score': auc_score,
    }

    model.to(current_device)
    return model_info

def preparation_embedding(texts):
    encodings = tokenizer(texts, padding=True, truncation=True, return_tensors="pt")
    input_ids = encodings['input_ids']
    token_type_ids = encodings['token_type_ids']
    embeded = embedding_model(input_ids=input_ids, token_type_ids=token_type_ids)
    return embeded

def GAN_step(optimizerG, optimizerD, netG, netD, real_data, label, epoch, i):
    netD.zero_grad()
    batch_size = real_data.size(0)

    output = netD(real_data)
    errD_real = criterion(output, label)
    errD_real.backward()
    D_x = output.mean().item()

    noise = torch.randn(batch_size, nz, device=device)
    fake_data = netG(noise).last_hidden_state
    label.fill_(1)
    output = netD(fake_data.detach())
    errD_fake = criterion(output, label)
    errD_fake.backward()
    D_G_z1 = output.mean().item()
    errD = errD_real + errD_fake
    optimizerD.step()

    netG.zero_grad()
    label.fill_(0)
    output = netD(fake_data)
    errG = criterion(output, label)
    errG.backward()
    D_G_z2 = output.mean().item()
    optimizerG.step()
    if i % 50 == 0:
        print('[%d/%d][%d/%d] Loss_D: %.4f Loss_G: %.4f D(x): %.4f D(G(z)): %.4f / %.4f'
              % (epoch, num_epochs, i, len(train_loader), errD.item(), errG.item(), D_x, D_G_z1, D_G_z2))

    return optimizerG, optimizerD, netG, netD


In [ ]:
netG = Generator(nz).to(device)
netD = Discriminator().to(device)

criterion = nn.BCELoss()
optimizerD = optim.Adam(netD.parameters(), lr=lr, betas=(beta1, 0.999))
optimizerG = optim.Adam(netG.parameters(), lr=lr, betas=(beta1, 0.999))

model_infos = []
for epoch in range(num_epochs):
    for i, data in enumerate(train_loader, 0):
        with torch.no_grad():
            embeded = preparation_embedding(data[0])

        optimizerG, optimizerD, netG, netD = GAN_step(
            optimizerG=optimizerG,
            optimizerD=optimizerD,
            netG=netG,
            netD=netD,
            real_data=embeded.to(device),
            label=data[1].float().to(device),
            epoch=epoch, i=i)

    auc_score = eval_auc(netD)
    model_infos.append(get_model_info_dict(netD, epoch, auc_score))

print('Train complete！')

[0/30][0/620] Loss_D: 1.4828 Loss_G: 0.6292 D(x): 0.4455 D(G(z)): 0.4198 / 0.4669
[0/30][50/620] Loss_D: 1.5972 Loss_G: 0.6658 D(x): 0.5351 D(G(z)): 0.5064 / 0.4818
[0/30][100/620] Loss_D: 1.4684 Loss_G: 0.6823 D(x): 0.4995 D(G(z)): 0.4901 / 0.4899
[0/30][150/620] Loss_D: 1.3027 Loss_G: 0.7460 D(x): 0.5136 D(G(z)): 0.5741 / 0.5072
[0/30][200/620] Loss_D: 1.5155 Loss_G: 0.6584 D(x): 0.5725 D(G(z)): 0.5408 / 0.4823
[0/30][250/620] Loss_D: 1.6197 Loss_G: 0.9005 D(x): 0.5659 D(G(z)): 0.4747 / 0.5496
[0/30][300/620] Loss_D: 1.3602 Loss_G: 0.9201 D(x): 0.5339 D(G(z)): 0.5611 / 0.5587
[0/30][350/620] Loss_D: 1.6030 Loss_G: 0.7826 D(x): 0.5745 D(G(z)): 0.4985 / 0.5402
[0/30][400/620] Loss_D: 1.4168 Loss_G: 0.6082 D(x): 0.5091 D(G(z)): 0.5093 / 0.4425
[0/30][450/620] Loss_D: 1.4224 Loss_G: 0.7642 D(x): 0.5466 D(G(z)): 0.5367 / 0.5341
[0/30][500/620] Loss_D: 1.5029 Loss_G: 0.7074 D(x): 0.5274 D(G(z)): 0.4914 / 0.5054
[0/30][550/620] Loss_D: 1.3252 Loss_G: 0.8258 D(x): 0.5435 D(G(z)): 0.5848 / 0.

# Inference

In [ ]:
max_auc_model_info = max(model_infos, key=lambda x: x['auc_score'])
model = Discriminator()
model.load_state_dict(max_auc_model_info['model_state_dict'])
model.to(device)
model.eval()

In [ ]:
class InferenceDataset(torch.utils.data.Dataset):
    def __init__(self, texts):
        self.texts = texts

    def __getitem__(self, idx):
        return self.texts[idx]

    def __len__(self):
        return len(self.texts)

sub_dataset = InferenceDataset(src_sub.text.to_list())

In [ ]:
inference_loader = DataLoader(sub_dataset, batch_size=2, shuffle=False)

sub_predictions = []
with torch.no_grad():
    for batch in inference_loader:
        encodings = tokenizer(batch, padding=True, truncation=True, return_tensors="pt")
        input_ids = encodings['input_ids']
        token_type_ids = encodings['token_type_ids']
        embeded = embedding_model(input_ids=input_ids, token_type_ids=token_type_ids)
        embeded = embeded.to(device)

        outputs = model(embeded)
        sub_predictions.extend(outputs.cpu().numpy())

sub_ans_df = pd.DataFrame.from_dict({'id':src_sub.id, 'generated': sub_predictions})
print(sub_ans_df)